In [1]:
import html2text
import numpy as np
import pandas as pd

In [2]:
composition_path: str = "./data/raw/composition.json"
description_path: str = "./data/raw/description.json"
price_path: str = "./data/raw/prices.csv"
product_out_path: str = "./data/processed/product.csv"

# COMPOSITION

In [3]:
def explode_ingredients(dict_list: list[dict[str, str]] | float) -> str:
    if isinstance(dict_list, float):
        return dict_list
    if not dict_list:
        return None
    return ", ".join([item["title"] for item in dict_list])

def explode_plainingredients(string: str) -> str:
    if isinstance(string, float) or string is None:
        return string
    return ", ".join(string.split("\n"))

def extract_sample(sample: list[list[dict[str, str | float | int]]]) -> dict[str, str | float | int]:
    if not sample:
        return {}
    return {k: v["amount"] for k, v in sample[0]["values"].items()}

In [4]:
composition_df = pd.read_json(composition_path).T.dropna(how="all")

In [5]:
composition_df.shape

(11377, 5)

In [6]:
composition_df.head()

,productId,nutritionalValues,ingredients,plainIngredients,allergens
92839,92839,[],[],Sodium Tallowate (A)/ Sodium Palmate (B)*\nAqu...,None
67672,67672,[],"[{'type': 'component', 'ingredients': [], 'tit...",None,"{'contained': [], 'possiblyContained': []}"
92838,92838,[],[],Sodium Tallowate (A)/ Sodium Palmate (B)*\nAqu...,None
85734,85734,[],"[{'type': 'component', 'ingredients': [], 'tit...",None,"{'contained': [], 'possiblyContained': []}"
85736,85736,[],"[{'type': 'component', 'ingredients': [], 'tit...",None,"{'contained': [], 'possiblyContained': []}"


In [7]:
composition_df["clean_ingredients"] = [explode_ingredients(f) for f in composition_df["ingredients"].tolist()]
composition_df["clean_plain_ingredients"] = [explode_plainingredients(f) for f in composition_df["plainIngredients"].tolist()]

In [8]:
composition_df["unified_ingredients"] = np.where(
    pd.isnull(composition_df["clean_ingredients"]),
    composition_df["clean_plain_ingredients"],
    composition_df["clean_ingredients"]
)

In [9]:
composition_df[composition_df["unified_ingredients"].isnull()]

,productId,nutritionalValues,ingredients,plainIngredients,allergens,clean_ingredients,clean_plain_ingredients,unified_ingredients


In [10]:
composition_df["value_dict"] = [
    extract_sample(
        [f for f in sample if f["portion"] == "100 g"]) 
    for sample in composition_df["nutritionalValues"].tolist()]

In [11]:
composition_df = composition_df.join(composition_df["value_dict"].apply(pd.Series))

In [12]:
composition_df["allergen_contain"] = [f.get("contained", []) if isinstance(f, dict) else [] for f in composition_df["allergens"]]

In [13]:
composition_df["allergen_possibly_contain"] = [f.get("possiblyContained", []) if isinstance(f, dict) else [] for f in composition_df["allergens"]]

In [14]:
drop_composition_cols = [
    "nutritionalValues", "ingredients", "plainIngredients", "allergens",
    "clean_ingredients", "clean_plain_ingredients", "value_dict"
]

In [15]:
composition_df = composition_df.drop(drop_composition_cols, axis=1).set_index("productId")

In [16]:
composition_df.head()

,unified_ingredients,energyKJ,energyKCal,fats,saturatedFats,carbohydrates,sugars,protein,salt,fiber,allergen_contain,allergen_possibly_contain
productId,,,,,,,,,,,,
92839,"Sodium Tallowate (A)/ Sodium Palmate (B)*, Aqu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[]
67672,"Szappanosított pálmaolaj, Szappanosított kókus...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[]
92838,"Sodium Tallowate (A)/ Sodium Palmate (B)*, Aqu...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[]
85734,"sodium lauroyl isethionate, stearic acid, laur...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[]
85736,"aqua, sodium lauroyl isethionate, stearic acid...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[]


# DESCRIPTION

In [17]:
description_df = pd.read_json(description_path).T

In [18]:
description_df.shape

(12599, 2)

In [19]:
description_df.head()

,productId,description
92839,92839,"<div class=""ckContent""><p style=""text-align:ju..."
67672,67672,"<div class=""ckContent""><p style=""text-align:ju..."
92838,92838,"<div class=""ckContent""><p style=""text-align:ju..."
85734,85734,"<div class=""ckContent""><p style=""text-align:ju..."
85736,85736,"<div class=""ckContent""><p style=""text-align:ju..."


In [20]:
description_df["description"] = [
    html2text.html2text(text) if isinstance(text, str) else text for text in description_df["description"].tolist()]

In [21]:
description_df = description_df.set_index("productId")
description_df

,description
productId,
92839,A lanolinos Baba szappan a Baba márka legrégeb...
67672,A szilárd szappan nemcsak **gyengéden tisztítj...
92838,A lanolinos Baba szappan a Baba márka legrégeb...
85734,"Fontos, hogy napi teendőid közepette is szánj ..."
85736,"Ha bőröd mostanában száraznak érzed, biztosan ..."
...,...
91720,**Kitchin egész hámozott paradicsom paradicsom...
97852,**Kitchin Zöldborsó**\n\nFedezd fel a frissess...
104806,**Bonduelle egész gomba mini üveges (280 g)**\...


# PRICE

In [22]:
prices = pd.read_csv(price_path).drop("cat", axis=1).drop_duplicates().set_index("prod")

In [23]:
prices

,price
prod,
92839,989
67672,699
92838,279
85734,569
85736,1499
...,...
91720,369
97852,609
104806,1049


# UNIFY

In [24]:
product_df = description_df.join(
    composition_df, how="outer").join(prices, how="left")

In [25]:
product_df.shape

(12652, 14)

In [26]:
product_df = product_df.dropna(how="all", subset=[f for f in product_df.columns if not f == "price"])

In [27]:
product_df.shape

(12648, 14)

In [28]:
product_df.to_csv(product_out_path)